# SQL Mastery Course - Advanced Level

Analytical SQL, optimization, and business intelligence.

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('..\sql_mastery.db')
def run_query(q): return pd.read_sql_query(q, conn)
print("Connected!")

Connected!


## Monthly Revenue Analysis

In [ ]:
run_query("""SELECT STRFTIME('%Y-%m', order_date) AS month, COUNT(*) AS orders, ROUND(SUM(total_amount), 2) AS revenue FROM orders WHERE status = 'Delivered' GROUP BY month ORDER BY month;""")

## Moving Average & Running Total

In [ ]:
run_query("""SELECT order_id, order_date, total_amount, SUM(total_amount) OVER (ORDER BY order_date) AS running_total, ROUND(AVG(total_amount) OVER (ORDER BY order_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3 FROM orders WHERE status = 'Delivered' ORDER BY order_date;""")

## Customer Lifetime Value

In [ ]:
run_query("""SELECT c.customer_id, c.first_name || ' ' || c.last_name AS name, COUNT(DISTINCT o.order_id) AS orders, ROUND(SUM(o.total_amount), 2) AS total_spent, ROUND(AVG(o.total_amount), 2) AS avg_order FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id ORDER BY total_spent DESC;""")

## RFM Segmentation

In [ ]:
run_query("""WITH rfm AS (SELECT c.customer_id, c.first_name || ' ' || c.last_name AS name, JULIANDAY('2024-06-01') - JULIANDAY(MAX(o.order_date)) AS recency, COUNT(DISTINCT o.order_id) AS frequency, ROUND(SUM(o.total_amount), 2) AS monetary FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id), scores AS (SELECT *, NTILE(4) OVER (ORDER BY recency DESC) AS r, NTILE(4) OVER (ORDER BY frequency) AS f, NTILE(4) OVER (ORDER BY monetary) AS m FROM rfm) SELECT name, recency, frequency, monetary, r, f, m, CASE WHEN r>=3 AND f>=3 AND m>=3 THEN 'Champion' WHEN r>=3 AND f>=2 THEN 'Loyal' WHEN r>=3 AND f=1 THEN 'New' WHEN r<=2 AND f>=3 THEN 'At Risk' ELSE 'Lost' END AS segment FROM scores ORDER BY monetary DESC;""")

## EXPLAIN QUERY PLAN

In [ ]:
run_query("EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 5;")

In [ ]:
# Category performance with window functions
run_query("""SELECT p.category, COUNT(DISTINCT p.product_id) AS products, COUNT(oi.quantity) AS units_sold, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue, ROUND(SUM(oi.quantity * oi.unit_price) * 100.0 / SUM(SUM(oi.quantity * oi.unit_price)) OVER (), 1) AS revenue_pct FROM products p JOIN order_items oi ON p.product_id = oi.product_id GROUP BY p.category ORDER BY revenue DESC;""")

In [ ]:
conn.close()
print("Done! Ready for the capstone project.")